# TinyFace alignment benchmark — local Jupyter
Chạy trực tiếp trong kernel Jupyter/GPU local. Không dùng Google Colab API hoặc Modal.

In [ ]:
from pathlib import Path
import os,sys,subprocess
ROOT=Path.cwd()
if not (ROOT/'CVLface-main').is_dir(): ROOT=Path('/content/Landmark-aligmenttest')
REPO=ROOT/'CVLface-main'; DATA=REPO/'tinyface'; RESULTS=ROOT/'results'; CACHE=ROOT/'model_cache'
for p in (DATA,RESULTS,CACHE): p.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(REPO/'cvlface'))
print('ROOT',ROOT)


In [ ]:
# Dependencies; restart kernel only if pip requests it
pkgs=['gdown','numpy==1.26.4','torchvision','transformers==4.34.1','huggingface-hub','omegaconf','opencv-python-headless','onnxruntime-gpu','insightface==0.7.3','mediapipe==0.10.14','pandas','pillow','tqdm','scikit-image','tabulate']
subprocess.check_call([sys.executable,'-m','pip','install','-q']+pkgs)


In [ ]:
# Download/unzip TinyFace
FILE_ID='1xTZc7lNmWN33ECO2AKH6FycGdiqIK7W0'; ZIP=ROOT/'tinyface.zip'
if not (DATA/'Testing_Set/Probe').is_dir():
 import gdown,zipfile,tempfile,shutil
 out=gdown.download(id=FILE_ID,output=str(ZIP),quiet=False)
 if not out: raise RuntimeError('Drive file must be Anyone with the link')
 with tempfile.TemporaryDirectory() as td:
  with zipfile.ZipFile(ZIP) as z:z.extractall(td)
  src=next((p for p in [Path(td),*Path(td).rglob('*')] if p.is_dir() and (p/'Testing_Set/Probe').is_dir()),None)
  if src is None:raise RuntimeError('ZIP missing Testing_Set/Probe')
  if DATA.exists():shutil.rmtree(DATA)
  shutil.copytree(src,DATA)
for s in ('Probe','Gallery_Match','Gallery_Distractor'):print(s,len(list((DATA/'Testing_Set'/s).glob('*.jpg'))))


In [ ]:
# Load recognizer and four alignment pipelines
import json,time,cv2,numpy as np,torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download
from general_utils.huggingface_model_utils import load_model_from_local_path
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type!='cuda':raise RuntimeError('GPU CUDA is required for full TinyFace')
def model(repo):
 p=CACHE/repo.replace('/','__')
 if not (p/'config.json').exists():snapshot_download(repo_id=repo,local_dir=str(p),token=os.getenv('HF_TOKEN'))
 return load_model_from_local_path(str(p)).to(DEVICE).eval()
recognizer=model('minchul/cvlface_adaface_vit_base_kprpe_webface4m')
dfa={'dfa_mobilenet':model('minchul/cvlface_DFA_mobilenet'),'dfa_resnet50':model('minchul/cvlface_DFA_resnet50')}
from insightface.app import FaceAnalysis
scrfd=FaceAnalysis(name='buffalo_l',root=str(CACHE/'insightface'),allowed_modules=['detection'],providers=['CUDAExecutionProvider','CPUExecutionProvider']);scrfd.prepare(ctx_id=0,det_thresh=.3,det_size=(640,640))
import mediapipe as mp
mesh=mp.solutions.face_mesh.FaceMesh(static_image_mode=True,max_num_faces=1,refine_landmarks=False,min_detection_confidence=.3)
ARC=np.array([[38.2946,51.6963],[73.5318,51.5014],[56.0252,71.7366],[41.5493,92.3655],[70.7299,92.2041]],np.float32)
def square(p):
 x=np.asarray(Image.open(p).convert('RGB'));h,w=x.shape[:2];s=max(h,w);y=np.zeros((s,s,3),np.uint8);y[(s-h)//2:(s-h)//2+h,(s-w)//2:(s-w)//2+w]=x;return y
def ten(x):return torch.from_numpy(x.copy()).permute(2,0,1).float().div(127.5).sub(1).unsqueeze(0).to(DEVICE)
def warp(x,k):
 M,_=cv2.estimateAffinePartial2D(k.astype(np.float32),ARC,method=cv2.LMEDS)
 if M is None:raise RuntimeError('transform failed')
 y=cv2.warpAffine(x,M,(112,112));ak=np.c_[k,np.ones(5)]@M.T
 return ten(y),torch.tensor(ak/112,dtype=torch.float32,device=DEVICE)[None]
def infer(path,name):
 x=square(path)
 if name in dfa:
  o=dfa[name](ten(x));return o[0],o[2],float(o[3].item())
 if name=='scrfd10g':
  fs=scrfd.get(cv2.cvtColor(x,cv2.COLOR_RGB2BGR))
  if not fs:raise RuntimeError('no face')
  f=max(fs,key=lambda z:(z.bbox[2]-z.bbox[0])*(z.bbox[3]-z.bbox[1]));a,k=warp(x,np.asarray(f.kps,np.float32));return a,k,float(f.det_score)
 r=mesh.process(cv2.resize(x,(640,640),interpolation=cv2.INTER_CUBIC))
 if not r.multi_face_landmarks:raise RuntimeError('no face mesh')
 p=r.multi_face_landmarks[0].landmark
 def m(ii):return np.mean([[p[i].x*x.shape[1],p[i].y*x.shape[0]] for i in ii],0)
 a,k=warp(x,np.asarray([m([33,133]),m([362,263]),m([1]),m([61]),m([291])],np.float32));return a,k,1.
def embedding(a,k):return F.normalize(recognizer(a,k).float(),dim=1)[0].cpu().numpy().astype('float32')
print('models loaded on',DEVICE)


In [ ]:
# Preflight then full run with resume every 100 images
splits={'probe':sorted((DATA/'Testing_Set/Probe').glob('*.jpg')),'gallery_match':sorted((DATA/'Testing_Set/Gallery_Match').glob('*.jpg')),'distractor':sorted((DATA/'Testing_Set/Gallery_Distractor').glob('*.jpg'))}
items=[(p,s) for s,ps in splits.items() for p in ps];PIPELINES=['dfa_mobilenet','dfa_resnet50','scrfd10g','mediapipe']
def label(p):return int(p.stem.split('_')[0])
for n in PIPELINES:
 with torch.inference_mode():a,k,c=infer(items[0][0],n);e=embedding(a,k)
 assert a.shape[-2:]==(112,112) and k.shape==(1,5,2) and e.shape==(512,);print(n,'preflight OK')
def run(name):
 cp=RESULTS/f'{name}.npz';rows=[];start=0
 if cp.exists():
  z=np.load(cp,allow_pickle=True);rows=z['rows'].tolist();start=int(z['next']);print('resume',name,start)
 with torch.inference_mode():
  for i,(p,s) in enumerate(tqdm(items[start:],desc=name),start):
   t=time.perf_counter();r={'path':str(p),'split':s,'label':label(p) if s!='distractor' else -100,'ok':False,'error':'','embedding':None}
   try:a,k,c=infer(p,name);r.update(ok=c>=.3,confidence=c,embedding=embedding(a,k))
   except Exception as ex:r['error']=f'{type(ex).__name__}: {ex}'
   r['latency_ms']=(time.perf_counter()-t)*1000;rows.append(r)
   if (i+1)%100==0 or i+1==len(items):np.savez_compressed(cp,rows=np.array(rows,dtype=object),next=i+1)
 return rows
results={n:run(n) for n in PIPELINES}


In [ ]:
# Metrics and reports
import pandas as pd
def metric(rows):
 good=[r for r in rows if r['ok'] and r['embedding'] is not None];by={r['path']:r['embedding'] for r in good};gal=[r for r in rows if r['split']!='probe' and r['path'] in by];G=np.stack([by[r['path']] for r in gal]);gl=np.array([r['label'] for r in gal]);ranks={1:[],5:[],20:[]}
 for p in [r for r in rows if r['split']=='probe']:
  hit=np.zeros(len(G),bool) if p['path'] not in by else gl[np.argsort(-(G@by[p['path']]))]==p['label']
  for q in ranks:ranks[q].append(float(hit[:q].any()))
 lat=[r['latency_ms'] for r in rows];return {'coverage':np.mean([r['ok'] for r in rows]),'rank1':100*np.mean(ranks[1]),'rank5':100*np.mean(ranks[5]),'rank20':100*np.mean(ranks[20]),'p50_ms':np.percentile(lat,50),'p95_ms':np.percentile(lat,95)}
summary=pd.DataFrame([{'pipeline':n,**metric(r)} for n,r in results.items()]);summary.to_csv(RESULTS/'summary.csv',index=False);(RESULTS/'results.json').write_text(json.dumps({n:metric(r) for n,r in results.items()},indent=2));(RESULTS/'benchmark_report.md').write_text('# TinyFace local benchmark\n\nSingle-view, no flip TTA.\n\n'+summary.to_markdown(index=False));display(summary);print(RESULTS)
